In [2]:
import pandas as pd
from pathlib import Path
import os
from dotenv import dotenv_values

In [3]:
data_path = Path(os.path.abspath('')).resolve().parent / "data.csv"
env_path = Path(os.path.abspath('')).resolve().parent / "secrets.env"

df = pd.read_csv(data_path)

env_vars = dotenv_values(env_path)
# Filter out bad coordinates

df = df.dropna(subset = ['longitude', 'latitude'])
df = df[(df['longitude'] != 0) & (df['latitude'] != 0)]

df['street_name'] = df['street_name'].apply(lambda x : x.title())

df['address'] = df.apply(lambda x : str(x['house_number']) + ' ' + x['street_name'] ,axis = 1)
print(df.shape)
df.head()

(970, 26)


,inspection_type,job_ticket_or_work_order_id,job_id,job_progress,bbl,boro_code,block,lot,house_number,street_name,...,location,community_board,council_district,census_tract,bin,nta,approved_date,x_coord,y_coord,address
0,BAIT,3056322,PC8487803,5,3.019150e+09,3,1915,61,284,Clinton Avenue,...,"{'latitude': '40.689887042515', 'longitude': '...",2.0,35.0,195.0,3055028.0,Clinton Hill,NaN,NaN,NaN,284 Clinton Avenue
1,Initial,14208272,PC8635827,1,3.030980e+09,3,3098,19,172,Seigel Street,...,"{'latitude': '40.704790525696', 'longitude': '...",1.0,34.0,489.0,3071470.0,East Williamsburg,2025-12-09T14:52:51.000,NaN,NaN,172 Seigel Street
2,BAIT,3056363,PC8548734,7,3.051220e+09,3,5122,10,116,East 19 Street,...,"{'latitude': '40.646503808371', 'longitude': '...",14.0,40.0,512.0,3117591.0,Flatbush,NaN,994915.0,174820.0,116 East 19 Street
3,BAIT,3056271,PC8612711,6,1.021080e+09,1,2108,52,469,West 157 Street,...,"{'latitude': '40.832524213959', 'longitude': '...",12.0,10.0,239.0,1062508.0,Washington Heights (South),2025-12-11T11:30:29.000,1000516.0,242596.0,469 West 157 Street
4,BAIT,3056414,PC8609532,4,2.033570e+09,2,3357,248,415,East 204 Street,...,"{'latitude': '40.870789389041', 'longitude': '...",7.0,11.0,425.0,2018686.0,Norwood,2025-12-09T09:20:49.000,NaN,NaN,415 East 204 Street


In [24]:
test_row = df.iloc[0]

for row in df.head(10).iterrows():
    latitude = row[1]['latitude']
    longitude = row[1]['longitude']
    address = row[1]['address']
    print(latitude, longitude, address)

40.689887042515 -73.968177844567 284 Clinton Avenue
40.704790525696 -73.939378259695 172 Seigel Street
40.646503808371 -73.961568071532 116 East   19 Street
40.832524213959 -73.94122003788 469 West  157 Street
40.870789389041 -73.87651098883 415 East 204 Street
40.767160410349 -73.897685854609 71-11 Astoria Boulevard N
40.790615419424 -73.948436619708 122 East 103 Street
40.71526031998 -73.984420123573 460 Grand Street
40.682307685049 -73.964644481108 554 Washington Avenue
40.682307685049 -73.964644481108 554 Washington Avenue


In [6]:
print(results)

NameError: name 'results' is not defined

In [26]:
import requests
import json

# Los tacos lat/lon

# lat = 40.7065340451609
# lon = -74.0116639196217

# Bk restaurant example
lat = 40.689425
lon = -73.9723664

# Demo
                # "latitude": 37.7937,
                # "longitude": -122.3965

# Replace with your actual API Key
API_KEY = env_vars['GOOGLE_PLACES_API_KEY']

# The endpoint for the Nearby Search (New) API
url = "https://places.googleapis.com/v1/places:searchNearby"

# Define the request body as a Python dictionary
# This example searches for restaurants within a 500-meter radius
# around a specific latitude and longitude.
request_body = {
    "includedTypes": ["restaurant", "meal_delivery", "meal_takeaway", "bakery", "bar", "cafe"],
    "maxResultCount": 20,
    "locationRestriction": {
        "circle": {
            "center": {
                "latitude": lat,
                "longitude": lon
            },
            "radius": 10.0
        }
    }
}

# Define the headers for the request
# X-Goog-Api-Key is used for authentication.
# X-Goog-FieldMask specifies which fields to return in the response.
# It's good practice to only request the fields you need to save on processing time and billing.
headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "places.displayName,places.formattedAddress,places.location,places.rating"
}

try:
    # Make the POST request
    # The 'json' parameter in requests.post automatically serializes the dictionary to JSON
    # and sets the 'Content-Type' header to 'application/json'.
    response = requests.post(url, headers=headers, json=request_body)

    # Check if the request was successful (status code 200)
    if response.status_code == 200:
        response_data = response.json()
        print("Nearby Search Results:")
        if response_data and "places" in response_data:
            for place in response_data["places"]:
                display_name = place.get("displayName", {}).get("text", "N/A")
                formatted_address = place.get("formattedAddress", "N/A")
                location = place.get("location", {})
                rating = place.get("rating", "N/A")
                print(f"  Name: {display_name}")
                print(f"  Address: {formatted_address}")
                print(f"  Location: Latitude {location.get('latitude')}, Longitude {location.get('longitude')}")
                print(f"  Rating: {rating}")
                print("-" * 20)
        else:
            print("No places found or unexpected response format.")
    else:
        print(f"Error: Request failed with status code {response.status_code}")
        print(f"Response: {response.text}")

except requests.exceptions.RequestException as e:
    print(f"An error occurred: {e}")



Nearby Search Results:
  Name: Miss Ada
  Address: 184 DeKalb Ave, Brooklyn, NY 11205, USA
  Location: Latitude 40.689425, Longitude -73.9723664
  Rating: 4.6
--------------------


In [27]:
# pd.DataFrame(response_data['places'])
request_body['locationRestriction']['circle']['center']['latitude']
request_body['locationRestriction']['circle']['center']['longitude']

-73.9723664

In [58]:
import requests
import time
# Replace with your actual API Key
API_KEY = env_vars['GOOGLE_PLACES_API_KEY']

# The endpoint for the Nearby Search (New) API
url = "https://places.googleapis.com/v1/places:searchNearby"

# Define the request body as a Python dictionary
# This example searches for restaurants within a 500-meter radius
# around a specific latitude and longitude.
request_body = {
    "includedTypes": ["restaurant", "meal_delivery", "meal_takeaway", "bakery", "bar", "cafe"],
    "maxResultCount": 20,
    "locationRestriction": {
        "circle": {
            "center": {
                "latitude": 0,
                "longitude": 0
            },
            "radius": 25.0
        }
    }
}

headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "places.displayName,places.formattedAddress,places.location,places.rating"
}


requested_places = pd.DataFrame()


for row in df.loc[700 : 710, :].iterrows():

    temp_df = pd.DataFrame()

    latitude = row[1]['latitude']
    longitude = row[1]['longitude']
    address = row[1]['address']
    
    try:
        # Make the POST request
        # The 'json' parameter in requests.post automatically serializes the dictionary to JSON
        # and sets the 'Content-Type' header to 'application/json'.

        # Reset for every iteration
        request_body['locationRestriction']['circle']['center']['latitude'] = latitude
        request_body['locationRestriction']['circle']['center']['longitude'] = longitude

        # Make request
        response = requests.post(url, headers=headers, json=request_body)
        time.sleep(1)
        # Check if the request was successful (status code 200)
        if response.status_code == 200:
            response_data = response.json()
            print("Nearby Search Results:")
            if response_data and "places" in response_data:
                for place in response_data["places"]:
                    display_name = place.get("displayName", {}).get("text", "N/A")
                    formatted_address = place.get("formattedAddress", "N/A")
                    location = place.get("location", {})
                    rating = place.get("rating", "N/A")
                    print(f"  Name: {display_name}")
                    print(f"  Address: {formatted_address}")
                    print(f"  Location: Latitude {location.get('latitude')}, Longitude {location.get('longitude')}")
                    print(f"  Rating: {rating}")
                    print("-" * 20)
            else:
                print(response)
                print("No places found or unexpected response format.")
        else:
            print(f"Error: Request failed with status code {response.status_code}")
            print(f"Response: {response.text}")

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
    
    if response_data:
        temp_df = pd.DataFrame(response_data['places'])
        temp_df['latitude'] = latitude
        temp_df['longitude'] = longitude
        temp_df['address'] = address
        
        requested_places = pd.concat([requested_places, temp_df])
    else:
        print("Nothing returned!")
        print(response_data)

Nearby Search Results:
<Response [200]>
No places found or unexpected response format.
Nothing returned!
{}
Nearby Search Results:
  Name: TOKIODELIC
  Address: 177 Lafayette St, New York, NY 10013, USA
  Location: Latitude 40.7204061, Longitude -73.9985107
  Rating: 4.1
--------------------
  Name: Salmon Noodle 3.0
  Address: 177 Lafayette St, New York, NY 10013, USA
  Location: Latitude 40.7204061, Longitude -73.9985107
  Rating: 4.9
--------------------
Nearby Search Results:
<Response [200]>
No places found or unexpected response format.
Nothing returned!
{}
Nearby Search Results:
<Response [200]>
No places found or unexpected response format.
Nothing returned!
{}
Nearby Search Results:
<Response [200]>
No places found or unexpected response format.
Nothing returned!
{}
Nearby Search Results:
  Name: Norma Turtle Bay
  Address: 930 2nd Ave, New York, NY 10022, USA
  Location: Latitude 40.7545059, Longitude -73.96858859999999
  Rating: 4.7
--------------------
  Name: Simpl Coffee


In [59]:
requested_places

,formattedAddress,location,rating,displayName,latitude,longitude,address
0,"177 Lafayette St, New York, NY 10013, USA","{'latitude': 40.7204061, 'longitude': -73.9985...",4.1,"{'text': 'TOKIODELIC', 'languageCode': 'en'}",40.720523,-73.998662,178 Lafayette Street
1,"177 Lafayette St, New York, NY 10013, USA","{'latitude': 40.7204061, 'longitude': -73.9985...",4.9,"{'text': 'Salmon Noodle 3.0', 'languageCode': ...",40.720523,-73.998662,178 Lafayette Street
0,"930 2nd Ave, New York, NY 10022, USA","{'latitude': 40.7545059, 'longitude': -73.9685...",4.7,"{'text': 'Norma Turtle Bay', 'languageCode': '...",40.754480,-73.968793,930 2 Avenue
1,"927 2nd Ave, New York, NY 10022, USA","{'latitude': 40.7546081, 'longitude': -73.9689...",4.9,"{'text': 'Simpl Coffee', 'languageCode': 'en'}",40.754480,-73.968793,930 2 Avenue


In [57]:
requested_places.head(40)

,formattedAddress,location,rating,displayName,latitude,longitude,address
0,"1025 Manhattan Ave, Brooklyn, NY 11222, USA","{'latitude': 40.7339858, 'longitude': -73.955158}",4.5,"{'text': 'Wenwen', 'languageCode': 'en'}",40.733676,-73.955535,159 Green Street
1,"158 Green St, Brooklyn, NY 11222, USA","{'latitude': 40.7336453, 'longitude': -73.9551...",4.9,"{'text': 'The Blue Light Speak Cheesy', 'langu...",40.733676,-73.955535,159 Green Street
2,"150 Green St, Brooklyn, NY 11222, USA","{'latitude': 40.7334873, 'longitude': -73.9555...",4.4,"{'text': 'ILIS', 'languageCode': 'en'}",40.733676,-73.955535,159 Green Street
3,"1015 Manhattan Ave, Brooklyn, NY 11222, USA","{'latitude': 40.733551299999995, 'longitude': ...",4.6,"{'text': 'Bombay Grill house', 'languageCode':...",40.733676,-73.955535,159 Green Street
4,"1017 Manhattan Ave, Brooklyn, NY 11222, USA","{'latitude': 40.7336111, 'longitude': -73.955}",4.1,"{'text': 'Triangulo Pizzeria', 'languageCode':...",40.733676,-73.955535,159 Green Street
5,"1011 Manhattan Ave, Brooklyn, NY 11222, USA","{'latitude': 40.733475899999995, 'longitude': ...",5.0,"{'text': 'Piscator', 'languageCode': 'en'}",40.733676,-73.955535,159 Green Street
0,"398 Knickerbocker Ave, Brooklyn, NY 11237, USA","{'latitude': 40.6998373, 'longitude': -73.9206...",4.4,"{'text': 'Ugly Donuts & Corn Dogs Bushwick', '...",40.699514,-73.920435,221 Himrod Street
0,"398 Knickerbocker Ave, Brooklyn, NY 11237, USA","{'latitude': 40.6998373, 'longitude': -73.9206...",4.4,"{'text': 'Ugly Donuts & Corn Dogs Bushwick', '...",40.699619,-73.920334,225 Himrod Street
0,"234 Irving Ave, Brooklyn, NY 11237, USA","{'latitude': 40.700337399999995, 'longitude': ...",4.8,"{'text': 'LA PITAYA DELI GRILL INC', 'language...",40.700171,-73.917448,1375 Greene Avenue
0,"234 Irving Ave, Brooklyn, NY 11237, USA","{'latitude': 40.700337399999995, 'longitude': ...",4.8,"{'text': 'LA PITAYA DELI GRILL INC', 'language...",40.700100,-73.917517,1371 Greene Avenue


In [ ]:
df.loc[10 : 100, :]

,inspection_type,job_ticket_or_work_order_id,job_id,job_progress,bbl,boro_code,block,lot,house_number,street_name,...,location,community_board,council_district,census_tract,bin,nta,approved_date,x_coord,y_coord,address
10,BAIT,3056367,PC8577767,5,3.051220e+09,3,5122,1,1800,Albermarle Road,...,"{'latitude': '40.647009146651', 'longitude': '...",14.0,40.0,512.0,3117590.0,Flatbush,2025-12-08T15:19:59.000,994665.0,175004.0,1800 Albermarle Road
11,BAIT,3056239,PC8641500,1,2.027520e+09,2,2752,32,1175,Vyse Avenue,...,"{'latitude': '40.828009878589', 'longitude': '...",3.0,17.0,12101.0,2006173.0,Crotona Park East,NaN,NaN,NaN,1175 Vyse Avenue
12,BAIT,3056413,PC8337315,8,2.033570e+09,2,3357,32,3158,Webster Avenue,...,"{'latitude': '40.87228977812', 'longitude': '-...",7.0,11.0,425.0,2018668.0,Norwood,2025-12-09T09:20:35.000,NaN,NaN,3158 Webster Avenue
13,Compliance,14208791,PC8617662,2,1.003930e+09,1,393,18,622,East 11 Street,...,"{'latitude': '40.727064616994', 'longitude': '...",3.0,2.0,28.0,1004882.0,East Village,2025-12-09T14:58:39.000,NaN,NaN,622 East 11 Street
14,BAIT,3056394,PC8463589,9,1.021530e+09,1,2153,68,554,West 181 Street,...,"{'latitude': '40.84894689122', 'longitude': '-...",12.0,10.0,269.0,1079910.0,Washington Heights (North),2025-12-09T07:23:06.000,NaN,NaN,554 West 181 Street
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,Initial,14209494,PC8634733,1,1.019180e+09,1,1918,3,2265,Adam C Powell Blvd,...,"{'latitude': '40.814201942049', 'longitude': '...",10.0,9.0,226.0,1000000.0,Harlem (North),2025-12-16T16:27:39.000,NaN,NaN,2265 Adam C Powell Blvd
97,Initial,14208858,PC8636377,1,2.024330e+09,2,2433,70,1061,Teller Avenue,...,"{'latitude': '40.828960548053', 'longitude': '...",4.0,16.0,175.0,2002241.0,Concourse-Concourse Village,2025-12-09T13:13:35.000,NaN,NaN,1061 Teller Avenue
98,BAIT,3056387,PC8591577,4,1.011280e+09,1,1128,61,331,Columbus Avenue,...,"{'latitude': '40.779714323075', 'longitude': '...",7.0,6.0,161.0,1028789.0,Upper West Side (Central),2025-12-09T07:28:34.000,NaN,NaN,331 Columbus Avenue
99,BAIT,3056410,PC8337190,8,2.033500e+09,2,3350,25,350,East 207 Street,...,"{'latitude': '40.875052991949', 'longitude': '...",7.0,11.0,42901.0,2018447.0,Norwood,2025-12-09T09:17:41.000,NaN,NaN,350 East 207 Street


In [24]:
requested_places_saved = requested_places.copy()

requested_places_saved

""


In [ ]:
import pickle

pickle.